In [1]:
from google.colab import files
uploaded = files.upload()


Saving BIClean.xlsx to BIClean.xlsx


In [4]:
import pandas as pd

# baca file excel
df = pd.read_excel("BIClean.xlsx")

# cek 5 baris pertama
df.head()


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-120.46,38.91,33,1561,282,30,11,1875,500001,INLAND
1,-119.31,33.82,16,343,85,29,14,21042,87500,INLAND
2,-123.01,38.80,34,1988,511,36,15,4625,162500,INLAND
3,-117.64,40.27,25,958,245,28,16,2625,67500,INLAND
4,-120.00,40.12,14,388,108,35,17,61359,106300,INLAND


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# --- 1. Pisahkan variabel X dan y ---
X = df[['longitude','latitude','housing_median_age','total_rooms',
        'total_bedrooms','population','households','median_income']]
y = df['median_house_value']

# --- 2. Bagi data train & test ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 3. Buat model regresi ---
model = LinearRegression()
model.fit(X_train, y_train)

# --- 4. Prediksi ---
y_pred = model.predict(X_test)

# --- 5. Evaluasi ---
print("Intercept:", model.intercept_)
print("Koefisien:", model.coef_)
print("MSE:", mean_squared_error(y_test, y_pred))
print("R²:", r2_score(y_test, y_pred))


Intercept: 351178.1746394498
Koefisien: [ 2.26465609e+02 -8.23511016e+03  1.93756946e+03  1.48329443e+01
 -5.89363858e+01 -8.17248322e+01  2.56820167e+02  2.90892848e+00]
MSE: 7522696110.7377405
R²: 0.3975915148402368


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Baca dataset

df = pd.read_excel("BIClean.xlsx")

# 2. Pisahkan X dan y
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]

# 3. Tentukan kolom kategorikal
categorical_cols = ["ocean_proximity"]
numeric_cols = X.drop(categorical_cols, axis=1).columns

# 4. Preprocessing: One-Hot Encoding untuk kategori
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop="first"), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

# 5. Buat pipeline regresi
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# 6. Split data train-test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 7. Latih model
model.fit(X_train, y_train)

# 8. Prediksi
y_pred = model.predict(X_test)

# 9. Evaluasi
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("R²:", r2)


MSE: 6174128420.146119
R²: 0.5055831986283287


In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

# 1. Baca dataset
df = pd.read_excel("BIClean.xlsx")

# 2. Pisahkan X dan y
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]

# 3. Tentukan kolom kategorikal
categorical_cols = ["ocean_proximity"]
numeric_cols = X.drop(categorical_cols, axis=1).columns

# 4. Preprocessing: One-Hot Encoding untuk kategori
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop="first"), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

# 5. Buat model Linear Regression
regressor = LinearRegression()

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', regressor)
])

# 6. Split data train-test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 7. Latih model
model.fit(X_train, y_train)

# 8. Prediksi
y_pred = model.predict(X_test)

# 9. Evaluasi
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("R²:", r2)

# 10. Ambil nama fitur setelah preprocessing
ohe = model.named_steps['preprocessor'].named_transformers_['cat']
ohe_features = ohe.get_feature_names_out(categorical_cols)
all_features = np.concatenate([ohe_features, numeric_cols])

# 11. Ambil intercept & koefisien
intercept = model.named_steps['regressor'].intercept_
coefficients = model.named_steps['regressor'].coef_

print("\nPersamaan Regresi:")
equation = f"y = {intercept:.2f} "
for feat, coef in zip(all_features, coefficients):
    equation += f"+ ({coef:.2f} * {feat}) "
print(equation)


MSE: 6174128420.146119
R²: 0.5055831986283287

Persamaan Regresi:
y = 175930.13 + (-94014.98 * ocean_proximity_INLAND) + (103571.64 * ocean_proximity_ISLAND) + (2192.51 * ocean_proximity_NEAR BAY) + (2088.45 * ocean_proximity_NEAR OCEAN) + (12.87 * longitude) + (-1684.77 * latitude) + (1022.39 * housing_median_age) + (27.82 * total_rooms) + (-42.81 * total_bedrooms) + (-76.45 * population) + (144.63 * households) + (2.25 * median_income) 
